# Applicant Field Consistency Validation with Gemini (Vertex AI)

This notebook:

- Authenticates to Google Cloud using `GOOGLE_APPLICATION_CREDENTIALS` (service account JSON).
- Reads a DataFrame with columns:  
  `['applicant_id', 'document_type', 'kpi_description', 'field_name', 'value']`
- Groups rows by `applicant_id`.
- For each applicant, sends all their records to **Gemini 2.5 Flash** via Vertex AI.
- Asks the model to:
  - Use both `field_name` **and** `kpi_description` to understand what each field means.
  - Treat semantically equivalent KPIs as the same underlying field (e.g., *Borrower Name* vs *Applicant Full Name*).
  - Mark each record as `validated = true/false` depending on consistency with other equivalent fields.
- Aggregates all model outputs into a single DataFrame.
- Saves the final result as JSON to `../data/validated_applicant_fields.json`.


In [40]:
import os
from pathlib import Path

# === Configure Google Cloud credentials ===
# Update this path to point to your local service account JSON.
# Example: ../../turing-agent-358210-a38a4820a9ce.json

SERVICE_ACCOUNT_JSON = Path("../../turing-agent-358210-a38a4820a9ce.json")

if not SERVICE_ACCOUNT_JSON.exists():
    raise FileNotFoundError(f"Service account JSON not found at: {SERVICE_ACCOUNT_JSON}")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(SERVICE_ACCOUNT_JSON.resolve())
print("Using GOOGLE_APPLICATION_CREDENTIALS:", os.environ["GOOGLE_APPLICATION_CREDENTIALS"])

Using GOOGLE_APPLICATION_CREDENTIALS: D:\FIU\Capstone II\turing-agent-358210-a38a4820a9ce.json


In [41]:
import json
import re
from typing import Any, Dict, List

import pandas as pd
import vertexai
from vertexai.generative_models import GenerativeModel
from pathlib import Path

# === Vertex AI configuration ===
PROJECT_ID = "turing-agent-358210"  # change if needed
LOCATION = "us-central1"            # or your chosen region

vertexai.init(project=PROJECT_ID, location=LOCATION)

GEMINI_MODEL_NAME = "gemini-2.5-flash"
model = GenerativeModel(GEMINI_MODEL_NAME)

print("Vertex AI initialized.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Model:", GEMINI_MODEL_NAME)

Vertex AI initialized.
Project: turing-agent-358210
Location: us-central1
Model: gemini-2.5-flash


c:\Users\luisd\anaconda3\envs\capstone_env\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [42]:
# === Load input data ===
# Expected columns:
# ['applicant_id', 'document_type', 'kpi_description', 'field_name', 'value']

INPUT_PATH = Path("../data/gemini_llm_kpi_anymatch_extracted.csv")  # update if needed
OUTPUT_PATH = Path("../data/validated_applicant_fields.csv")

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input file not found at: {INPUT_PATH}")

if INPUT_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(INPUT_PATH)
elif INPUT_PATH.suffix.lower() in {".parquet", ".pq"}:
    df = pd.read_parquet(INPUT_PATH)
else:
    raise ValueError(f"Unsupported input format: {INPUT_PATH.suffix}")

expected_cols = ['applicant_id', 'document_type', 'kpi_description', 'field_name', 'value']
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df['applicant_id'] = df['applicant_id'].astype(str)

df = df[expected_cols].copy()
df.head()

,applicant_id,document_type,kpi_description,field_name,value
0,1,APPLICATION FORM_46,External FI reports /RCU Report,RCU Check Date,25 DEC 2023
1,1,APPLICATION FORM_46,Loan Application Form,Document Type,Loan Application Form
2,1,DEVIATION APPROVAL MAILS_34,Deviation Approval Mails,DeviationApprovalOutcome,Approved with 0.50% PF
3,1,DISBURSEMENT MEMO_43,Disbursement memo,Document Type,DISBURSEMENT MEMO
4,1,DISBURSEMENT MEMO_43,NACH and PDC,Payment Mode,NACH (E-NACH)


In [43]:
# === Helper: build per-applicant JSON payload ===

def build_applicant_payload(df_applicant: pd.DataFrame) -> Dict[str, Any]:
    """Build a JSON-serializable payload for a single applicant.

    {
        "applicant_id": <id>,
        "records": [
            {
                "applicant_id": ...,
                "document_type": ...,
                "kpi_description": ...,
                "field_name": ...,
                "value": ...
            },
            ...
        ]
    }
    """
    df_applicant = df_applicant.sort_values(
        ["field_name", "document_type", "kpi_description"]
    ).reset_index(drop=True)

    records = df_applicant.to_dict(orient="records")
    applicant_id = df_applicant["applicant_id"].iloc[0]

    return {
        "applicant_id": applicant_id,
        "records": records,
    }


In [44]:
# === Helper: extract JSON from model response ===
import re
import json

def extract_json_from_text(raw_text: str):
    """
    Extract a JSON object/array from model response text.

    - Handles optional Markdown code fences (``` or ```json).
    - Tries to slice from the first '{' or '[' to the last '}' or ']'.
    - Cleans common LLM artifacts like trailing commas and // comments.
    """
    text = raw_text.strip()

    # If there are code fences, just strip the backticks and any leading "json"
    if text.startswith("```"):
        # remove leading ```... and trailing ```
        text = re.sub(r"^```[a-zA-Z]*", "", text)
        text = re.sub(r"```$", "", text).strip()

    # Find the first '{' or '[' and the last '}' or ']'
    start_candidates = [i for i in [text.find("{"), text.find("[")] if i != -1]
    end_candidates = [text.rfind("}"), text.rfind("]")]
    end_candidates = [i for i in end_candidates if i != -1]

    if not start_candidates or not end_candidates:
        raise ValueError(f"Could not find JSON object/array in response:\n{raw_text}")

    start = min(start_candidates)
    end = max(end_candidates) + 1
    json_str = text[start:end]

    # Remove JS-style comments if the model added any
    json_str = re.sub(r"//.*", "", json_str)

    # Remove trailing commas before closing brackets/braces
    json_str = re.sub(r",(\s*[\]}])", r"\\1", json_str)

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print("=== DEBUG: cleaned JSON candidate ===")
        print(json_str[:1000])
        print("====================================")
        raise ValueError(
            "Failed to parse JSON from model response after cleaning. "
            "See DEBUG log above."
        ) from e

In [45]:
VALIDATION_INSTRUCTION = """You are a data validation assistant for a loan underwriting team.

You receive JSON data for a SINGLE applicant. Each item in `records` is a field extracted
from one document and contains:

- applicant_id
- document_type
- kpi_description  (a human-readable description of what the field means)
- field_name
- value

IMPORTANT:
- The same real-world field (e.g., applicant name, SSN, home address, employer)
  may appear with slightly different `field_name` values in different documents.
- Use BOTH `field_name` AND `kpi_description` to determine when two records represent
  the SAME underlying concept.

Examples of equivalent concepts:
- "Borrower Name" ~ "Applicant Full Name"
- "SSN" ~ "Social Security Number"
- "Home Addr" ~ "Residential Address"
- "Employer Name" ~ "Current Company"

YOUR TASK:
For this SINGLE applicant, determine whether each record's `value` is consistent
with other records that represent the same underlying concept.

You must return a JSON ARRAY where each element corresponds 1:1 to an input record,
with EXACTLY these fields:

- "applicant_id"  (string or number)
- "document_type" (string)
- "field_name"    (string)
- "value"         (string)
- "validated"     (boolean)

Rules for `validated`:
- true  -> This value is consistent with other semantically equivalent fields
           for the same applicant (either identical or clearly equivalent).
- false -> There is a clear conflict, a major discrepancy, or not enough evidence
           to be confident it matches.

CRITICAL:
- Do NOT add extra fields.
- Do NOT drop any records.
- The number of objects in the output array MUST equal the number of input records.
- The ENTIRE response MUST be valid JSON. Do NOT include code, comments, or Markdown.
"""


def validate_applicant_records(model: GenerativeModel, payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Send a single applicant payload to Gemini and return normalized results."""
    prompt = f"""{VALIDATION_INSTRUCTION}

Here is the applicant data as JSON:

{json.dumps(payload, ensure_ascii=False, default=str)}
"""

    # Force the model to respond with JSON only
    response = model.generate_content(
        [prompt],
        generation_config={"response_mime_type": "application/json"}
    )

    raw_text = response.text  # should be pure JSON now

    parsed = json.loads(raw_text)

    if isinstance(parsed, dict):
        parsed_list = [parsed]
    elif isinstance(parsed, list):
        parsed_list = parsed
    else:
        raise ValueError(f"Model returned JSON that is not an object or array: {type(parsed)}")

    required_keys = ["applicant_id", "document_type", "field_name", "value", "validated"]
    normalized: List[Dict[str, Any]] = []

    for item in parsed_list:
        if not isinstance(item, dict):
            continue

        norm = {k: item.get(k) for k in required_keys}

        # Coerce validated to bool if it's a string
        val = norm.get("validated")
        if isinstance(val, str):
            if val.lower() in {"true", "yes", "y", "1"}:
                norm["validated"] = True
            elif val.lower() in {"false", "no", "n", "0"}:
                norm["validated"] = False

        normalized.append(norm)

    return normalized


# test on a single applicant (uncomment when ready)
"""
first_applicant_id = '3' #df["applicant_id"].iloc[0]
sample_payload = build_applicant_payload(df[df["applicant_id"] == first_applicant_id])
sample_result = validate_applicant_records(model, sample_payload)
pd.DataFrame(sample_result).head()
"""


'\nfirst_applicant_id = \'3\' #df["applicant_id"].iloc[0]\nsample_payload = build_applicant_payload(df[df["applicant_id"] == first_applicant_id])\nsample_result = validate_applicant_records(model, sample_payload)\npd.DataFrame(sample_result).head()\n'

In [46]:
# === Main loop: validate all applicants ===

all_results: List[Dict[str, Any]] = []

for applicant_id, group in df.groupby("applicant_id"):
    payload = build_applicant_payload(group)
    print(f"Validating applicant_id={applicant_id} with {len(payload['records'])} records...")

    try:
        applicant_results = validate_applicant_records(model, payload)
        all_results.extend(applicant_results)
    except Exception as e:
        print(f"[ERROR] Validation failed for applicant_id={applicant_id}: {e}")
        # Fallback: mark all records as not validated for this applicant
        for _, row in group.iterrows():
            all_results.append(
                {
                    "applicant_id": row["applicant_id"],
                    "document_type": row["document_type"],
                    "field_name": row["field_name"],
                    "value": row["value"],
                    "validated": False,
                }
            )

len(all_results)

Validating applicant_id=1 with 171 records...
Validating applicant_id=2 with 106 records...
Validating applicant_id=3 with 167 records...
Validating applicant_id=4 with 80 records...


524

In [47]:
# === Build final DataFrame and save to ../data/ ===

validated_df = pd.DataFrame(all_results)

# Enforce final column order
final_cols = ["applicant_id", "document_type", "field_name", "value", "validated"]
validated_df = validated_df[final_cols]

# Ensure output directory exists
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

validated_df.to_csv(OUTPUT_PATH, index=False)

print("Saved validated results to:", OUTPUT_PATH.resolve())
validated_df.head()

Saved validated results to: D:\FIU\Capstone II\capstone\data\validated_applicant_fields.csv


,applicant_id,document_type,field_name,value,validated
0,1,VENDOR KYC AND BANK DETAILS_26,Account Name,Mr. SHIVARAJ PATIL,False
1,1,OCR DOCUMENTS_28,Account Number,918010058269238,False
2,1,VENDOR KYC AND BANK DETAILS_26,Account Number,00000064043649073,False
3,1,VENDOR KYC AND BANK DETAILS_26,Account Type,REGULAR SB CHQ-INDIVIDUALS,True
4,1,OCR DOCUMENTS_78,Amount Paid by Cheque 1,"10,00,000",False


In [48]:
validated_df.validated.value_counts()

validated
True     438
False     86
Name: count, dtype: int64